# Day 1 — ILT 4: Intro to Databricks + Lakeflow Connect vs Autoloader
**Duration:** 90 min &nbsp;|&nbsp; **Level:** Beginner &nbsp;|&nbsp; **Tags:** databricks, lakeflow-connect, autoloader, ingestion

**Time:** 2:00 PM – 3:30 PM  
**What comes next:** Hands-on — Connect to ADLS and ingest GlobalMart data using the patterns you learn here

---

## Learning Objectives

By the end of this session, students will be able to:

1. Explain what Databricks is and identify its key workspace components
2. Describe what Lakeflow Connect does and when to use it (Postgres CDC via WAL)
3. Describe what Autoloader does and when to use it (ADLS file drops)
4. Demonstrate Full Load and Incremental Load using Autoloader file ingestion
5. Choose the right tool for each GlobalMart table

---

### What we will cover today
1. What is Databricks and why Data Engineers use it
2. Key parts of the Databricks workspace
3. How a Spark cluster works (very simple)
4. The 2 Ingestion Tools — Lakeflow Connect (Postgres CDC) + Autoloader (file drops)
5. Full Load and Incremental Load patterns with live code

> **Instructor note:** This session is 90 minutes. Spend ~30 min on Databricks intro (show the UI live), then ~45 min on ingestion patterns with code. The last 15 min is wrap-up before hands-on.

## Section 1 — What is Databricks?

Databricks is a **cloud platform** built on top of **Apache Spark**.  
Think of it as a supercharged Jupyter notebook that can process millions of rows using many computers at the same time.

### Why do Data Engineers use Databricks?

| Problem | How Databricks Solves It |
|---------|-------------------------|
| Data is too big for one laptop | Uses many machines (a cluster) to share the work |
| Need to store data in a smart format | Uses Delta Lake (not just CSV files) |
| Need to schedule pipelines | Has built-in Workflows and Jobs |
| Need to manage code versions | Connects to GitHub via Databricks Repos |
| Need to govern who sees what data | Unity Catalog for permissions and lineage |

### Databricks vs Traditional Tools

```
Old way:  Laptop → Python script → CSV file on disk
New way:  Databricks cluster → PySpark → Delta Lake on Azure cloud
```

> The code looks almost the same as pandas, but it runs on 10 machines instead of 1.

## Section 2 — Key Parts of the Databricks Workspace

When you open Databricks, you see these sections on the left menu:

| Section | What it is | What we use it for |
|---------|-----------|--------------------|
| **Workspace** | Your notebook files | Write and run PySpark code |
| **Repos** | Connected to GitHub | Version control for notebooks |
| **Clusters** | The computing power | Must be running before you run code |
| **Jobs/Workflows** | Scheduled pipelines | Run notebooks automatically at a set time |
| **Data** | Browse tables and files | See Delta tables you have created |
| **SQL Editor** | SQL query tool | Query tables like a database |

### How a Cluster Works — Very Simple

```
  YOUR NOTEBOOK
       |
       v
   DRIVER NODE  ← This is the "brain" — it coordinates the work
   /    |    \
Worker Worker Worker  ← These are the "hands" — they process the data in parallel
```

- When you run a cell, the **Driver** reads your code and splits the data
- **Worker nodes** each process their part of the data
- Result comes back to the Driver and you see the output

> **For GlobalMart:** Our 10 CSV files are small, so we only need a small cluster.  
> In real companies, the data could be billions of rows — same code, bigger cluster.

## Section 3 — The 2 Ingestion Tools

**Ingestion** = reading data from a source and landing it in your Bronze Delta tables in ADLS.

GlobalMart uses **2 tools** to bring data into Bronze:

| # | Tool | Source Type | Trigger | Captures |
|---|------|-------------|---------|---------|
| 1 | **Autoloader** (`cloudFiles`) | Files in ADLS (`raw/`) | New file lands | New rows from new files |
| 2 | **Lakeflow Connect** | Supabase PostgreSQL | Continuous CDC | INSERT + UPDATE + DELETE |

---

### Tool 1: Autoloader

Autoloader watches a folder in ADLS. When new CSV/Parquet/JSON files arrive, it picks them up — without re-reading files it has already processed.

Within Autoloader ingestion, you use two loading strategies depending on the table type:

| Strategy | What it reads | Write mode | Use for |
|----------|---------------|------------|---------|
| **Full Load** | All rows, every run | overwrite | Small lookup tables (rarely change) |
| **Incremental Load** | Only new rows (watermark) | append | Large tables (grow daily) |

### Tool 2: Lakeflow Connect

Lakeflow Connect reads the **PostgreSQL Write-Ahead Log (WAL)** — a log of every database change — continuously. Every INSERT, UPDATE, and DELETE at the source is captured as a CDC event and delivered to Bronze automatically without full table scans.

> We explore Lakeflow Connect as a live CDC POC in **Day 5**. Today we understand the concept.

---

### GlobalMart — Which tool for which table?

| Tables | Tool | Reason |
|--------|------|--------|
| `orders`, `order_items` | **Lakeflow Connect** | Live transactional — updated continuously, need full CDC |
| `payment_methods`, `shipping_tier`, `products`, `suppliers` | **Autoloader — Full Load** | Small lookup tables, rarely change |
| `customers`, `payments`, `addresses`, `returns` | **Autoloader — Incremental** | Grow every day, track new rows by date |

> In today's hands-on, we will practise **Full Load** and **Incremental Load** using file-based ingestion (Autoloader pattern).

## Setup — Connect to ADLS Gen2

Run this cell first. Every cell below depends on the variables set here.

In [ ]:
# ─── ADLS Connection Setup ───────────────────────────────────────────────────
# Azure Portal → Storage Accounts → amazonprojectadls → Security + Networking → Access Keys

storage_account_name = "amazonprojectadls"
container_name       = "amazon-data"
raw_folder           = "raw"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"  # ← Replace this!

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

raw_path    = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{raw_folder}"
bronze_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"

print("Connected to ADLS!")
print(f"Raw path:    {raw_path}")
print(f"Bronze path: {bronze_path}")

## Pattern 1 — Full Load

**Rule:** Read ALL rows from the source file every time you run this.  
**Use for:** Small lookup tables like `payment_methods`, `shipping_tier`, `products`, `suppliers`.  
**Write mode:** `overwrite` — replaces whatever was there before. Safe to re-run.

```
Source CSV  ──read all rows──►  Bronze Delta Table
               (overwrite)
```

In [ ]:
# ─── Pattern 1: Full Load demo — payment_methods table ────────────────────────
# payment_methods is a small lookup table — only a few rows (Credit Card, UPI, etc.)
# Perfect for Full Load — read everything every time

# Step 1: Read all rows from the CSV
payment_methods_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{raw_path}/payment_methods.csv")

print(f"Total rows in payment_methods: {payment_methods_df.count()}")
payment_methods_df.show()

# Step 2: Write to Bronze using OVERWRITE
# OVERWRITE = delete what was there, write fresh — no duplicates possible
payment_methods_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_path}/payment_methods")

print("Full Load complete — payment_methods saved to Bronze!")

In [ ]:
# ─── Proof: run Full Load a second time — row count stays the same ─────────────
# This proves overwrite is safe to re-run (no duplicates)

payment_methods_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_path}/payment_methods")

count_after = spark.read.format("delta").load(f"{bronze_path}/payment_methods").count()
print(f"Row count after running Full Load TWICE: {count_after}")
print("Same number as before — overwrite prevents duplicates!")

In [ ]:
# ─── Full Load for all small/reference tables ─────────────────────────────────

full_load_tables = ["payment_methods", "shipping_tier", "products", "suppliers"]

for table_name in full_load_tables:
    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(f"{raw_path}/{table_name}.csv")

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{bronze_path}/{table_name}")

    print(f"Full Load done: {table_name} ({df.count()} rows)")

print("\nAll reference tables loaded!")

## Pattern 2 — Incremental Load

**Rule:** Read only the rows that are NEW since the last time you ran.  
**Use for:** Large tables that grow every day — `orders`, `orders_items`, `payments`, `returns`.  
**Write mode:** `append` — adds only new rows, does not touch existing rows.

### What is a Watermark?

A watermark is the **date of the last record you already saved**.  
Each run, you ask: *"give me only rows that arrived AFTER my watermark."*

```
Run 1 — loads orders from 2016 + 2017   → watermark saved = 2017-12-31
Run 2 — loads orders from 2018 onwards  → only NEW rows, appended to Bronze
```

### How we demo this using orders.csv

The `orders.csv` already has orders across **multiple years**.  
We split the same file by date to simulate two pipeline runs — no second file needed.

```
orders.csv  ┌─ 2016 + 2017 orders ─┐  ← Run 1 loads these ("old data")
            └─ 2018 orders ─────────┘  ← Run 2 loads these ("new data")
```

**Watch the Bronze row count grow after each run — that is incremental loading.**

In [ ]:
# ─── Step 1: Explore orders.csv — find the date column automatically ──────────

from pyspark.sql.functions import col, to_timestamp, year, min as spark_min, max as spark_max

orders_raw = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(f"{raw_path}/orders.csv")

print("All columns in orders.csv:")
orders_raw.printSchema()

print("Sample rows:")
orders_raw.show(3, truncate=False)

# Auto-detect which column looks like a date — any column with 'date' or 'time' in its name
date_candidates = [c for c in orders_raw.columns if "date" in c.lower() or "time" in c.lower()]
print(f"Columns that look like dates: {date_candidates}")

# Use the first match found automatically
# If it picks the wrong one, change DATE_COLUMN = "your_column_name" manually here
DATE_COLUMN = date_candidates[0] if date_candidates else None
print(f"Using date column: '{DATE_COLUMN}'")

# Cast to proper timestamp so we can filter and group by year
orders_df = orders_raw.withColumn("_order_ts", to_timestamp(col(DATE_COLUMN)))

# Show the full date range in the file
orders_df.select(
    spark_min("_order_ts").alias("earliest_order"),
    spark_max("_order_ts").alias("latest_order")
).show(truncate=False)

# Show how many orders exist per year — this tells us how to split the data
print("Orders count by year:")
orders_df.withColumn("order_year", year("_order_ts")) \
    .groupBy("order_year").count().orderBy("order_year").show()

# Auto-calculate cutoff: the last year in the data = Run 2, everything before = Run 1
max_year = orders_df.withColumn("yr", year("_order_ts")).agg({"yr": "max"}).collect()[0][0]
CUTOFF_DATE = f"{max_year}-01-01"

print(f"Total orders in CSV   : {orders_df.count():,}")
print(f"Auto cutoff date      : {CUTOFF_DATE}")
print(f"Run 1 → orders BEFORE {CUTOFF_DATE}  (older data)")
print(f"Run 2 → orders FROM   {CUTOFF_DATE}  (newest year only)")

In [ ]:
# ─── Step 2: RUN 1 — First pipeline run ───────────────────────────────────────
# Load everything BEFORE the cutoff date — this is our "historical load"
# CUTOFF_DATE was calculated automatically in Step 1

run1_orders = orders_df.filter(col("_order_ts") < CUTOFF_DATE)

print(f"RUN 1: Orders before {CUTOFF_DATE}: {run1_orders.count():,} rows")
run1_orders.show(3, truncate=False)

# First write — OVERWRITE to start with a clean Bronze table
run1_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_path}/orders")

bronze_after_run1 = spark.read.format("delta").load(f"{bronze_path}/orders").count()
print(f"\nBronze orders AFTER RUN 1: {bronze_after_run1:,} rows")
print(f"Watermark for next run   : {CUTOFF_DATE}")
print("(In a real pipeline you would save this watermark to a control table)")

In [ ]:
# ─── Step 3: RUN 2 — New data has arrived ─────────────────────────────────────
# Use the same CUTOFF_DATE as the watermark to load ONLY the newest orders
# These are rows that did NOT exist when Run 1 ran

run2_orders = orders_df.filter(col("_order_ts") >= CUTOFF_DATE)

print(f"RUN 2: New orders from {CUTOFF_DATE} onwards: {run2_orders.count():,} rows")
run2_orders.show(3, truncate=False)

# APPEND — adds only the new rows on top of what Run 1 already wrote
# Does NOT delete or touch Run 1 data
run2_orders.write \
    .format("delta") \
    .mode("append") \
    .save(f"{bronze_path}/orders")

bronze_after_run2 = spark.read.format("delta").load(f"{bronze_path}/orders").count()
print(f"\nBronze orders AFTER RUN 1: {bronze_after_run1:,} rows")
print(f"Bronze orders AFTER RUN 2: {bronze_after_run2:,} rows")
print(f"Rows added by Run 2      : {bronze_after_run2 - bronze_after_run1:,} rows")

In [ ]:
# ─── Step 4: Prove it worked ──────────────────────────────────────────────────

total_csv    = orders_df.count()
bronze_total = spark.read.format("delta").load(f"{bronze_path}/orders").count()

print("─" * 50)
print("INCREMENTAL LOAD — SUMMARY")
print("─" * 50)
print(f"  Total rows in source CSV         : {total_csv:,}")
print(f"  Rows loaded in Run 1             : {run1_orders.count():,}")
print(f"  Rows loaded in Run 2             : {run2_orders.count():,}")
print(f"  Run 1 + Run 2 combined           : {run1_orders.count() + run2_orders.count():,}")
print(f"  Bronze row count now             : {bronze_total:,}")
print("─" * 50)

if bronze_total == total_csv:
    print("All rows accounted for — incremental load is correct!")
else:
    print(f"Difference: {abs(bronze_total - total_csv)} rows — check your cutoff date")

print()
print("Key insight: Run 2 loaded ONLY the new orders.")
print("The older orders were NOT re-read or re-written. That is incremental loading.")

## Full Load vs Incremental Load — Side by Side

| | Full Load | Incremental Load |
|-|-----------|------------------|
| **What it reads** | Every row, every run | Only new rows since last run |
| **Write mode** | `overwrite` | `append` |
| **Needs a watermark?** | No | Yes — save the max date after each run |
| **Safe to re-run?** | Yes — overwrite prevents duplicates | Only safe if watermark is correct |
| **Good for** | Small tables (payment_methods, shipping_tier) | Large tables (orders, orders_items, returns) |
| **Demo we just ran** | payment_methods (whole table, twice — same count) | orders (split by date — Bronze grew after Run 2) |

## Lakeflow Connect — CDC from Postgres

**Lakeflow Connect** is the Databricks-native tool for ingesting from databases using Change Data Capture (CDC).

For GlobalMart, it continuously reads changes from Supabase PostgreSQL (orders, order_items) via the Write-Ahead Log (WAL).

### What is CDC?

Without CDC, you would re-read the entire database every pipeline run.

With CDC, you only read what **changed** since the last run:

```
operation | order_id | status       | changed_at
----------+----------+--------------+------------------
INSERT    | O1001    | pending      | 2024-06-01 09:00
UPDATE    | O1001    | shipped      | 2024-06-02 14:30   ← status changed
UPDATE    | O1001    | delivered    | 2024-06-03 10:00   ← status changed again
INSERT    | O1002    | pending      | 2024-06-01 11:00
```

**Without CDC:** You would only know the current status of each order.  
**With CDC:** You have the full history of every state change — which matters for analytics.

### Lakeflow Connect vs Manual JDBC CDC

| | Manual JDBC + WAL | Lakeflow Connect |
|-|-------------------|-----------------|
| **Setup** | Write Python + SQL | GUI/config in Databricks |
| **WAL draining** | Must cache + drain slot manually | Handled automatically |
| **Continuous** | Must schedule manually | Built-in continuous mode |
| **Monitoring** | Custom alerts | Built-in pipeline monitoring |

> We implement CDC from Postgres in **Day 5 HOL** using the full WAL replication slot approach.  
> Lakeflow Connect is the production-grade alternative — same result, less code.

In [ ]:
# ─── Lakeflow Connect concept demo ───────────────────────────────────────────
# NOT running real Lakeflow Connect today — just showing what the output looks like
# Real Lakeflow Connect is configured as a pipeline in Day 5

from pyspark.sql import Row

# Sample of what Lakeflow Connect delivers to the Bronze streaming table
# Each row = one CDC event from Supabase Postgres (orders table)
sample_cdc_records = [
    Row(operation="INSERT", order_id="O1001", customer_id="C001", status="pending",   order_date="2024-06-01", _cdc_timestamp="2024-06-01 09:00:00"),
    Row(operation="UPDATE", order_id="O1001", customer_id="C001", status="shipped",   order_date="2024-06-01", _cdc_timestamp="2024-06-02 14:30:00"),
    Row(operation="UPDATE", order_id="O1001", customer_id="C001", status="delivered", order_date="2024-06-01", _cdc_timestamp="2024-06-03 10:00:00"),
    Row(operation="INSERT", order_id="O1002", customer_id="C002", status="pending",   order_date="2024-06-01", _cdc_timestamp="2024-06-01 11:00:00"),
]

lakeflow_df = spark.createDataFrame(sample_cdc_records)

print("Sample Lakeflow Connect output — CDC events delivered to Bronze:")
lakeflow_df.show(truncate=False)

print("Operations captured:")
lakeflow_df.groupBy("operation").count().show()

print()
print("Key insight: order O1001 has 3 rows here — INSERT + 2 UPDATEs.")
print("Silver layer will de-duplicate and keep only the LATEST status per order_id.")
print("Without CDC, you would only ever see 'delivered' and lose the status history.")

## Autoloader — Deep Dive

**Autoloader** (`cloudFiles`) is Databricks' incremental file ingestion tool. It is the standard way to ingest CSV, JSON, or Parquet files that land in ADLS over time.

### How Autoloader Works

```
New file lands in raw/
        ↓
Autoloader detects it (event notification or listing)
        ↓
Reads ONLY the new file — checkpoint records "already processed" files
        ↓
Writes to Bronze as Delta (format = "cloudFiles" → writes as Delta)
        ↓
Updates checkpoint → next run starts from here
```

### Two Trigger Modes

| Mode | When to use | How it runs |
|------|-------------|-------------|
| `trigger(availableNow=True)` | Scheduled pipelines — run once, process all new files | Runs to completion, then stops |
| `trigger(continuous=True)` | Always-on pipelines — process files as they land | Runs indefinitely until stopped |

For GlobalMart batch pipelines we use `availableNow=True` — it runs on schedule, processes the new files, and stops.

### Schema Evolution

One of Autoloader's biggest advantages: **it handles new columns automatically**.

If a file drop adds a new column (`supplier_rating`), Autoloader detects it, updates the schema, and adds the column to the Bronze Delta table.

Without Autoloader, a new column would crash the pipeline.

### Autoloader Code Pattern

```python
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", checkpoint_path + "schema")
    .option("header", "true")
    .load(raw_path + "customers_*.csv")
)

(
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("bronze.customers")
)
```

> We implement this hands-on in **Day 4**. Today — understand the concept: Autoloader = checkpoint-based file ingestion, no re-reads.

## Tool Comparison — Lakeflow Connect vs Autoloader

| Feature | Autoloader (`cloudFiles`) | Lakeflow Connect |
|---------|--------------------------|-----------------|
| **Source** | Files in ADLS (`raw/`) | PostgreSQL database |
| **Format** | CSV, JSON, Parquet | Tables (via WAL) |
| **Trigger** | New file lands | Continuous CDC |
| **Captures** | New rows from new files | INSERT + UPDATE + DELETE |
| **Write mode** | overwrite (Full Load) or append (Incremental) | Streaming table (append + MERGE) |
| **Schema evolution** | Yes — detects new columns automatically | Fixed schema from source |
| **Re-reads** | Never — checkpoint tracks progress | Never — WAL position tracked |
| **Bootcamp** | Day 1 patterns + Day 4 HOL | Day 5 HOL |

### GlobalMart Tool Assignment

```
Lakeflow Connect
    orders          → Bronze streaming table (INSERT + UPDATE + DELETE)
    order_items     → Bronze streaming table (INSERT + UPDATE + DELETE)

Autoloader (Full Load)
    payment_methods → Bronze Delta (overwrite — small, rarely changes)
    shipping_tier   → Bronze Delta (overwrite — small, rarely changes)
    products        → Bronze Delta (overwrite — catalog updates periodically)
    suppliers       → Bronze Delta (overwrite — small reference table)

Autoloader (Incremental Load)
    customers       → Bronze Delta (append — new customers arrive daily)
    payments        → Bronze Delta (append — new payments arrive daily)
    addresses       → Bronze Delta (append — grows over time)
    returns         → Bronze Delta (append — new returns arrive over time)
```

### Rule of Thumb

```
Database table, updated continuously?      → Lakeflow Connect
File drops in ADLS, rarely changes?       → Autoloader + Full Load (overwrite)
File drops in ADLS, grows daily?           → Autoloader + Incremental (append + watermark)
```

In [ ]:
# ─── List all 10 CSV files in ADLS and their recommended pattern ──────────────

files = dbutils.fs.ls(raw_path)

print("Files in ADLS raw folder:")
print("-" * 50)
for f in files:
    print(f"  {f.name:<35} {f.size / 1024:.1f} KB")

print(f"\nTotal files: {len(files)}")

# Recommended pattern for each actual file
pattern_map = {
    "payment_methods.csv": "Full Load    — small lookup table",
    "shipping_tier.csv":   "Full Load    — small lookup table",
    "products.csv":        "Full Load    — reference catalog",
    "suppliers.csv":       "Full Load    — reference table",
    "orders.csv":          "Incremental  — grows every day",
    "orders_items.csv":    "Incremental  — grows with every order",
    "payments.csv":        "Incremental  — grows with every order",
    "returns.csv":         "Incremental  — new returns over time",
    "customers.csv":       "CDC          — rows get updated (email, city)",
    "addresses.csv":       "CDC          — addresses change over time",
}

print("\nRecommended ingestion pattern:")
print("-" * 60)
for table, pattern in pattern_map.items():
    print(f"  {table:<28} → {pattern}")

## Recap — What We Covered Today

| Topic | Key Takeaway |
|-------|--------------|
| What is Databricks | Cloud platform on Apache Spark — notebooks + cluster + Delta Lake + Jobs |
| Cluster structure | Driver (brain) + Workers (hands) — process data in parallel |
| **Autoloader** | Checkpoint-based file ingestion from ADLS — no re-reads, schema evolution |
| **Full Load** | Read all rows, overwrite destination — safe to re-run, no duplicates |
| **Incremental Load** | Read only new rows using a watermark — Bronze row count grows each run |
| **Lakeflow Connect** | Continuous CDC from Postgres via WAL — INSERT + UPDATE + DELETE captured |

---

## Hands-on Coming Next (3:30 PM)

You will:
1. Connect to ADLS with your storage key
2. Full Load — `payment_methods.csv` and `shipping_tier.csv` into Bronze
3. Incremental Load — `orders.csv` in two runs, watch the count grow
4. Verify Bronze row count is correct and no duplicates exist

> Code cells in the Full Load and Incremental Load sections are your template — use that code directly.

---

## Coming Up in the Bootcamp

| Day | Topic |
|-----|-------|
| **Day 2** | Delta Lake deep dive — ACID, transaction log, time travel |
| **Day 4** | Autoloader HOL — live file ingestion from ADLS into Bronze |
| **Day 5** | Lakeflow Connect + CDC POC — Postgres WAL into Bronze |
| **Day 6+** | Bronze → Silver transformation, SCD, dimensional modelling |

> The Bronze tables you build today are exactly what we will deep-dive on tomorrow.